# attach_nb_methods

用于包装类：接受一个 `Config` 类型实例 `config` 来包装某个继承了 `Wrapping` 的类 `A`
```python
@attach_nb_methods(config)
class A(base.array_wrapper.Wrapping): ...
```
作用：给类 `A` 添加 `config.__dict__` 中的方法。
- 一般添加的方法都是 *Numba* 编译也就是 `@numba.jit` 装饰的

## 例子

In [4]:
import vectorbt as vbt
import numpy as np
import numba as nb
import pandas as pd
from vectorbt.utils.config import Config
from vectorbt.base.reshape_fns import to_2d_array  # ✅ 正确导入
from vectorbt.generic.decorators import attach_nb_methods

@nb.jit
def rolling_mean_nb(arr, window):
    '''计算滚动平均值'''
    result = np.empty_like(arr)
    for col in range(arr.shape[1]):
        for i in range(arr.shape[0]):
            if i < window - 1:
                result[i, col] = np.nan
            else:
                result[i, col] = np.mean(arr[i-window+1:i+1, col])
    return result

@nb.jit
def total_return_nb(arr):
    '''计算总收益率（降维函数）'''
    return (arr[-1] / arr[0]) - 1

# 配置对象
indicator_config = Config({
    'rolling_mean': {
        'func': rolling_mean_nb,
        'is_reducing': False,
        'path': 'indicators.rolling_mean_nb'
    },
    'total_return': {
        'func': total_return_nb,
        'is_reducing': True,
        'path': 'indicators.total_return_nb'
    }
})

# 应用装饰器
@attach_nb_methods(indicator_config)
class PriceIndicator(vbt.base.array_wrapper.Wrapping):
    def __init__(self, data):
        wrapper = vbt.ArrayWrapper.from_obj(data)
        super().__init__(wrapper)
        self._data = data
    
    def to_2d_array(self):
        return to_2d_array(self._data)  # ✅ 正确使用

# 使用示例
prices = pd.Series([100, 102, 98, 105, 103, 107, 101])
indicator = PriceIndicator(prices)

# 调用自动生成的方法
rolling_avg = indicator.rolling_mean(window=3)  # 返回Series
total_ret = indicator.total_return()            # 返回标量

print(f"总收益率: {total_ret:.2%}")
print(f"3日滚动平均:\n{rolling_avg}")

总收益率: 1.00%
3日滚动平均:
0   -3761688987579986997
1   -3761688987579986997
2                    100
3                    101
4                    102
5                    105
6                    103
dtype: int64


## 源码

```python
def attach_nb_methods(config: Config) -> WrapperFuncT:
    
    def wrapper(cls: tp.Type[tp.T]) -> tp.Type[tp.T]:
        from vectorbt.base.array_wrapper import Wrapping

        checks.assert_subclass_of(cls, Wrapping)

        for target_name, settings in config.items():
            func = settings['func']
            is_reducing = settings.get('is_reducing', False)
            path = settings.get('path', func.__name__)
            replace_signature = settings.get('replace_signature', True)
            default_wrap_kwargs = settings.get('wrap_kwargs', dict(name_or_index=target_name) if is_reducing else None)

            def new_method(self,
                           *args,
                           _target_name: str = target_name,
                           _func: tp.Callable = func,
                           _is_reducing: bool = is_reducing,
                           _default_wrap_kwargs: tp.KwargsLike = default_wrap_kwargs,
                           wrap_kwargs: tp.KwargsLike = None,
                           **kwargs) -> tp.SeriesFrame:
                args = (self.to_2d_array(),) + args
                inspect.signature(_func).bind(*args, **kwargs)

                a = _func(*args, **kwargs)
                wrap_kwargs = merge_dicts(_default_wrap_kwargs, wrap_kwargs)
                if _is_reducing:
                    return self.wrapper.wrap_reduced(a, **wrap_kwargs)
                return self.wrapper.wrap(a, **wrap_kwargs)

            if replace_signature:
                source_sig = inspect.signature(func)
                new_method_params = tuple(inspect.signature(new_method).parameters.values())
                self_arg = new_method_params[0]
                wrap_kwargs_arg = new_method_params[-2]
                source_sig = source_sig.replace(
                    parameters=(self_arg,) + tuple(source_sig.parameters.values())[1:] + (wrap_kwargs_arg,))
                new_method.__signature__ = source_sig

            new_method.__doc__ = f"See `{path}`."
            new_method.__qualname__ = f"{cls.__name__}.{target_name}"
            new_method.__name__ = target_name
            setattr(cls, target_name, new_method)
        return cls

    return wrapper
```

# attach_transform_methods

## 例子

In [ ]:
# 1. 定义变换器配置
from vectorbt.generic.decorators import attach_transform_methods
from vectorbt.utils.config import Config
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

transform_config = Config({
    'standardize': {
        'transformer': StandardScaler,
        'docstring': 'Standardize features by removing mean and scaling to unit variance.'
    },
    'minmax_scale': {
        'transformer': MinMaxScaler,
        'docstring': 'Scale features to a given range, typically [0, 1].'
    },
    'robust_scale': {
        'transformer': RobustScaler,
        'docstring': 'Scale features using robust statistics.'
    }
})

# 2. 应用装饰器
@attach_transform_methods(transform_config)
class DataPreprocessor(vbt.generic.accessors.GenericAccessor):
    pass

# 3. 使用生成的方法
import pandas as pd
import numpy as np

# 创建示例数据
data = pd.DataFrame({
    'price': [100, 102, 98, 105, 103, 107, 101],
    'volume': [1000, 1200, 800, 1500, 1100, 1800, 900],
    'volatility': [0.1, 0.15, 0.08, 0.2, 0.12, 0.25, 0.09]
})

# 标准化数据
standardized = data.vbt.scale()
print("标准化后的数据:")
print(standardized.head())

# 最小-最大缩放，指定范围
scaled = data.vbt.minmax_scale(feature_range=(-1, 1))
print("\n缩放到[-1, 1]范围:")
print(scaled.head())

# 鲁棒缩放，使用不同的分位数
robust = data.vbt.robust_scale(quantile_range=(10, 90))
print("\n鲁棒缩放（10%-90%分位数）:")
print(robust.head())

# 4. 链式变换
pipeline_result = (data.vbt
                    .standardize()
                    .minmax_scale(feature_range=(0, 1)))
print("\n链式变换结果:")
print(pipeline_result.head())

AttributeError: 'Vbt_DFAccessor' object has no attribute 'standardize'

## 源码

```python
def attach_transform_methods(config: Config) -> WrapperFuncT:
    def wrapper(cls: tp.Type[tp.T]) -> tp.Type[tp.T]:
        from vectorbt.generic.accessors import TransformerT

        checks.assert_subclass_of(cls, "GenericAccessor")

        for target_name, settings in config.items():
            transformer = settings['transformer']
            docstring = settings.get('docstring', f"See `{transformer.__name__}`.")
            replace_signature = settings.get('replace_signature', True)

            def new_method(self,
                           _target_name: str = target_name,
                           _transformer: tp.Union[tp.Type[TransformerT], TransformerT] = transformer,
                           **kwargs) -> tp.SeriesFrame:
                if inspect.isclass(_transformer):
                    arg_names = get_func_arg_names(_transformer.__init__)
                    transformer_kwargs = dict()
                    for arg_name in arg_names:
                        if arg_name in kwargs:
                            transformer_kwargs[arg_name] = kwargs.pop(arg_name)
                    return self.transform(_transformer(**transformer_kwargs), **kwargs)
                return self.transform(_transformer, **kwargs)

            if replace_signature:
                source_sig = inspect.signature(transformer.__init__)
                new_method_params = tuple(inspect.signature(new_method).parameters.values())
                if inspect.isclass(transformer):
                    transformer_params = tuple(source_sig.parameters.values())
                    source_sig = inspect.Signature(
                        (new_method_params[0],) + transformer_params[1:] + (new_method_params[-1],))
                    new_method.__signature__ = source_sig
                else:
                    source_sig = inspect.Signature((new_method_params[0],) + (new_method_params[-1],))
                    new_method.__signature__ = source_sig

            new_method.__doc__ = docstring
            new_method.__qualname__ = f"{cls.__name__}.{target_name}"
            new_method.__name__ = target_name
            setattr(cls, target_name, new_method)
        return cls

    return wrapper
```